In [ ]:
import sqlite3

conn = sqlite3.connect("law.db")
cursor = conn.cursor()

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load biến môi trường từ file .env
load_dotenv()

# Lấy API key
api_key = os.getenv("OPENAI_API_KEY")

# Tạo client
client = OpenAI(api_key=api_key)

In [ ]:
conn.close()

In [ ]:
def extract_from_db(id: int, include_parent=False):
    cursor.execute("SELECT * FROM laws WHERE id = ?", (id,))
    rows = cursor.fetchall()

    columns = [col[0] for col in cursor.description]
    results = [dict(zip(columns, row)) for row in rows]

    if include_parent:
        all_nodes = []
        for r in results:
            current = r
            while current['parent_id'] is not None:
                cursor.execute("SELECT * FROM laws WHERE id = ?", (current['parent_id'],))
                parent = cursor.fetchone()
                if parent is None:
                    break
                parent_dict = dict(zip(columns, parent))
                all_nodes.append(parent_dict)
                current = parent_dict
        results.extend(all_nodes)

    return results[::-1]

In [ ]:
rows = extract_from_db(166, include_parent=True)
law = ""
for r in rows:
    title = r['title']
    if not (title.strip().startswith("Chương") or title.strip().startswith("dâda") or title.strip().startswith("Điều")):
        law += r['title'] + r['content'] + "\n"
print(law)

In [ ]:
template1 = """Bạn là chuyên gia phân tích văn bản pháp luật và xây dựng knowledge graph. Nhiệm vụ của bạn là:

1. Phân tích đoạn văn bản luật pháp Việt Nam được cung cấp
2. Chỉnh sửa văn bản để làm rõ các khái niệm (concepts) và mối quan hệ (relations) mà KHÔNG làm thay đổi nội dung pháp lý
3. Trích xuất các bộ ba (triplet) theo format: Concept1 - Relation - Concept2

**HƯỚNG DẪN CHI TIẾT:**

- Chỉ thêm/sửa để làm rõ chủ ngữ, vị ngữ, tân ngữ
- Tách câu phức thành câu đơn nếu cần
- Làm rõ các đại từ (nó, họ, điều này...) bằng danh từ cụ thể
- Trong phần nội dung được gửi sẽ có 3 dòng, tập trung vào dòng thứ 3 (nội dung chính)

**YÊU CẦU VỀ TRIPLET:**
- Mỗi concept phải NGẮN GỌN (tối đa 3-5 từ)
- Ưu tiên danh từ/cụm danh từ cho concept
- Relations phải là ĐỘNG TỪ hoặc GIỚI TỪ đơn giản
- Tránh lặp lại thông tin không cần thiết
- Mỗi triplet phải độc lập và có ý nghĩa
- Ưu tiên trích xuất các triplet về mối quan hệ giữa các chủ thể trong luật pháp Việt Nam
- Bỏ qua mối quan hệ giữa các điều - khoản - điểm trong chung 1 item
**CÁC LOẠI RELATION THƯỜNG DÙNG:**
- cấm (prohibits)
- bị cấm (is_prohibited)
- yêu_cầu (requires)
- cho_phép (allows)
- áp_dụng_cho (applies_to)
- xảy_ra_tại (occurs_at)
- có_điều_kiện (has_condition)
- bao_gồm (includes)
- ngoại_trừ (excludes)


**YÊU CẦU OUTPUT:**
Không được trả lời bất kỳ điều gì ngoài phần OUTPUT dưới đây. Nếu không có triplet nào, chỉ cần trả về "Không có triplet nào".
- **ID:** [ID được cung cấp]
- **Item gốc:** [Văn bản gốc]
- **Item đã sửa:** [Văn bản sau khi chỉnh sửa để dễ trích xuất triplet]
- **Các bộ triplet:**
  1. [Concept1] - [Relation] - [Concept2]
  2. [Concept1] - [Relation] - [Concept2]
  ...

**VĂN BẢN CẦN PHÂN TÍCH:**
ID: {i}
Nội dung: {text}"""

In [ ]:
template = """Bạn là chuyên gia phân tích văn bản pháp luật và xây dựng knowledge graph. Nhiệm vụ của bạn là:

1. Phân tích đoạn văn bản luật pháp Việt Nam được cung cấp
2.  Trích xuất các bộ ba (triplet) theo format: Concept1 - Relation - Concept2

**HƯỚNG DẪN CHI TIẾT:**

- Tách câu phức thành câu đơn nếu cần
- Làm rõ các đại từ (nó, họ, điều này...) bằng danh từ cụ thể

**YÊU CẦU VỀ TRIPLET:**
- Mỗi concept phải NGẮN GỌN (tối đa 3-5 từ)
- Ưu tiên danh từ/cụm danh từ cho concept
- Relations phải là ĐỘNG TỪ hoặc GIỚI TỪ đơn giản
- Tránh lặp lại thông tin không cần thiết
- Mỗi triplet phải độc lập và có ý nghĩa
- Ưu tiên trích xuất các triplet về mối quan hệ giữa các chủ thể trong luật pháp Việt Nam
- Bỏ qua mối quan hệ giữa các điều - khoản - điểm trong chung 1 item
**CÁC LOẠI RELATION THƯỜNG DÙNG:**
- cấm (prohibits)
- bị cấm (is_prohibited)
- yêu_cầu (requires)
- cho_phép (allows)
- áp_dụng_cho (applies_to)
- xảy_ra_tại (occurs_at)
- có_điều_kiện (has_condition)
- bao_gồm (includes)
- ngoại_trừ (excludes)


**YÊU CẦU OUTPUT:**
Không được trả lời bất kỳ điều gì ngoài phần OUTPUT dưới đây. Nếu không có triplet nào, chỉ cần trả về "Không có triplet nào".
- **ID:** [ID được cung cấp]
- **Item gốc:** [Văn bản gốc]
- **Các bộ triplet:**
  1. [Concept1] - [Relation] - [Concept2]
  2. [Concept1] - [Relation] - [Concept2]
  ...

**VĂN BẢN CẦN PHÂN TÍCH:**
ID: {i}
Nội dung: {text}"""

In [ ]:
import pyperclip
import time

# Khởi tạo biến id (chạy cell này một lần)
if 'current_id' not in globals():
    current_id = 1
    print(f"Khởi tạo ID = {current_id}")

def next_prompt(i: int):
    global current_id
    
    while True:
        rows = extract_from_db(i, include_parent=True)
        
        # Kiểm tra nếu có title là "Khoản" hoặc "Điểm"
        has_item = any(r['title'].strip().startswith(("Khoản", "Điểm")) for r in rows)
        
        if has_item:
            # Nếu có Khoản/Điểm thì xử lý
            law = ""
            for r in rows:
                title = r['title']
                if not (title.strip().startswith("Chương") or title.strip().startswith("dâda")):
                    law +=r['title'] + " " + r['content'] + "\n"
            print(law)
            return template.format(i=i, text=law)
        else:
            # Nếu không có thì tăng current_id và lặp lại
            current_id += 1
            i = current_id


In [ ]:
current_id = 165

In [ ]:
# Cell này chạy mỗi lần bạn cần copy nội dung mới
def copy_next_content():
    global current_id

    # Lấy nội dung theo ID
    content = next_prompt(current_id)
    
    # Copy vào clipboard
    pyperclip.copy(content)
    
    # Hiển thị thông tin
    print(f"✅ Đã copy nội dung ID = {current_id}")
    
    print(f"➡️  ID tiếp theo sẽ là: {current_id + 1}")
    
    # Tăng ID cho lần sau
    current_id += 1
    
    return content

# Thực thi


In [24]:
copy_next_content()

Điều 15 Chuyển hướng xe
Khoản 4 Không được quay đầu xe ở phần đường dành cho người đi bộ qua đường, trên cầu, đầu cầu, gầm cầu vượt, ngầm, tại nơi đường bộ giao nhau cùng mức với đường sắt, đường hẹp, đường dốc, đoạn đường cong tầm nhìn bị che khuất, trên đường cao tốc, trong hầm đường bộ, trên đường một chiều, trừ khi có hiệu lệnh của người điều khiển giao thông hoặc chỉ dẫn của biển báo hiệu tạm thời.

✅ Đã copy nội dung ID = 179
➡️  ID tiếp theo sẽ là: 180


'Bạn là chuyên gia phân tích văn bản pháp luật và xây dựng knowledge graph. Nhiệm vụ của bạn là:\n\n1. Phân tích đoạn văn bản luật pháp Việt Nam được cung cấp\n2.  Trích xuất các bộ ba (triplet) theo format: Concept1 - Relation - Concept2\n\n**HƯỚNG DẪN CHI TIẾT:**\n\n- Tách câu phức thành câu đơn nếu cần\n- Làm rõ các đại từ (nó, họ, điều này...) bằng danh từ cụ thể\n\n**YÊU CẦU VỀ TRIPLET:**\n- Mỗi concept phải NGẮN GỌN (tối đa 3-5 từ)\n- Ưu tiên danh từ/cụm danh từ cho concept\n- Relations phải là ĐỘNG TỪ hoặc GIỚI TỪ đơn giản\n- Tránh lặp lại thông tin không cần thiết\n- Mỗi triplet phải độc lập và có ý nghĩa\n- Ưu tiên trích xuất các triplet về mối quan hệ giữa các chủ thể trong luật pháp Việt Nam\n- Bỏ qua mối quan hệ giữa các điều - khoản - điểm trong chung 1 item\n**CÁC LOẠI RELATION THƯỜNG DÙNG:**\n- cấm (prohibits)\n- bị cấm (is_prohibited)\n- yêu_cầu (requires)\n- cho_phép (allows)\n- áp_dụng_cho (applies_to)\n- xảy_ra_tại (occurs_at)\n- có_điều_kiện (has_condition)\n- bao_g

In [ ]:
user_input = copy_next_content()

# Gọi API, truyền biến user_input vào content
response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "user", "content": user_input}
    ]
)

# In kết quả trả lời từ assistant
print(response.choices[0].message.content)

In [ ]:
# Chạy cell này một lần để cài đặt
!pip install pyperclip
!pip install pandas  # nếu cần đọc từ file